# Financial Post 9 — Interactive Chart Annotation Tweaker

Run the chart generation, then tweak `ann_x` and `ann_y` per chart and re-run just that cell.

In [ ]:
# Setup: run once
import sys
sys.path.insert(0, '/Users/bob/LHM/Scripts/chart_generation')

# Import everything from the main script
from financial_edu_charts import *

# Force white theme
set_theme('white')

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Ready. Edit ann_x/ann_y in each cell and re-run to preview.')

In [ ]:
# Chart 1: HY OAS Percentile
# Tweak these values:
ann_x, ann_y = 0.65, 0.75

# --- generate chart inline ---
hy = fetch_db_level('BAMLH0A0HYM2', start='1997-01-01')
if len(hy) == 0: hy = fetch_fred_level('BAMLH0A0HYM2', start='1997-01-01')
hy_bps = hy * 100
win = min(5040, len(hy_bps))
p10 = hy_bps.rolling(win, min_periods=1000).quantile(0.10)
p25 = hy_bps.rolling(win, min_periods=1000).quantile(0.25)
p75 = hy_bps.rolling(win, min_periods=1000).quantile(0.75)
p90 = hy_bps.rolling(win, min_periods=1000).quantile(0.90)
last_val = hy_bps.iloc[-1]
pctile = (hy_bps < last_val).sum() / len(hy_bps) * 100

fig, ax = new_fig()
ax.fill_between(hy_bps.index, p10, p25, color=COLORS['port'], alpha=0.15)
ax.fill_between(hy_bps.index, p25, p75, color=COLORS['fog'], alpha=0.10)
ax.fill_between(hy_bps.index, p75, p90, color=COLORS['port'], alpha=0.15)
ax.plot(hy_bps.index, hy_bps, color=THEME['primary'], linewidth=2.0, label=f'HY OAS ({last_val:.0f} bps)', zorder=5)
style_single_ax(ax, fmt='{:.0f}')
set_xlim_to_data(ax, hy_bps.index)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
add_last_value_label(ax, hy_bps, color=THEME['primary'], fmt='{:.0f}', side='right')
add_recessions(ax, start_date='1997-01-01')
ax.legend(loc='upper right', fontsize=9, **legend_style())
add_annotation_box(ax, f'Current: {last_val:.0f} bps ({pctile:.0f}th pctile)\n20-year rolling distribution.')
brand_fig(fig, 'High-Yield Credit Spreads', subtitle='ICE BofA HY OAS | Percentile Distribution', source='ICE BofA via FRED', data_date=hy_bps.index[-1])
plt.show()

In [ ]:
# Chart 4: NFCI Decomposition
ann_x, ann_y = 0.82, 0.08

nfci = fetch_db_level('NFCI', start='1999-01-01')
risk = fetch_db_level('NFCIRISK', start='1999-01-01')
credit = fetch_db_level('NFCICREDIT', start='1999-01-01')
leverage = fetch_db_level('NFCILEVERAGE', start='1999-01-01')

fig, ax = new_fig()
ax.plot(risk.index, risk, color=COLORS['sky'], linewidth=1.5, alpha=0.7, label=f'Risk ({risk.iloc[-1]:.2f})')
ax.plot(credit.index, credit, color=COLORS['dusk'], linewidth=1.5, alpha=0.7, label=f'Credit ({credit.iloc[-1]:.2f})')
ax.plot(leverage.index, leverage, color=COLORS['sea'], linewidth=1.5, alpha=0.7, label=f'Leverage ({leverage.iloc[-1]:.2f})')
ax.plot(nfci.index, nfci, color=THEME['primary'], linewidth=2.5, label=f'NFCI Composite ({nfci.iloc[-1]:.2f})', zorder=5)
ax.axhline(0, color=COLORS['fog'], linewidth=0.8, linestyle='--', alpha=0.5)
style_single_ax(ax, fmt='{:.1f}')
set_xlim_to_data(ax, nfci.index)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
add_last_value_label(ax, nfci, color=THEME['primary'], fmt='{:.2f}', side='right')
add_recessions(ax, start_date='1999-01-01')
ax.legend(loc='upper right', fontsize=9, **legend_style())
add_annotation_box(ax, 'The headline hides the war.\nSubcomponents can diverge sharply.')
brand_fig(fig, 'NFCI Decomposition', subtitle='Composite vs Subindices', source='Chicago Fed via FRED', data_date=nfci.index[-1])
plt.show()

In [ ]:
# Chart 5: SLOOS vs Loan Growth
ann_x, ann_y = 0.50, 0.92

sloos = fetch_db_level('DRTSCILM', start='1990-01-01')
busloans = fetch_db_level('BUSLOANS', start='1988-01-01')
sloos_inv = sloos * -1
median_gap = busloans.index.to_series().diff().median().days
periods = 52 if median_gap < 10 else 12
busloans_yoy = (busloans.pct_change(periods) * 100).dropna()
busloans_yoy = busloans_yoy[busloans_yoy.index >= '1990-01-01']

fig, ax1 = new_fig()
ax2 = ax1.twinx()
c1, c2 = THEME['secondary'], THEME['primary']
ax1.plot(sloos_inv.index, sloos_inv, color=c1, linewidth=2.0, label=f'SLOOS Inv ({sloos_inv.iloc[-1]:.1f}%)')
ax2.plot(busloans_yoy.index, busloans_yoy, color=c2, linewidth=2.5, label=f'Loan Growth ({busloans_yoy.iloc[-1]:.1f}%)')
ax1.axhline(0, color=COLORS['fog'], linewidth=0.8, linestyle='--', alpha=0.5)
style_dual_ax(ax1, ax2, c1, c2)
set_xlim_to_data(ax1, sloos_inv.index)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
add_last_value_label(ax1, sloos_inv, color=c1, fmt='{:.1f}%', side='left')
add_last_value_label(ax2, busloans_yoy, color=c2, fmt='{:.1f}%', side='right')
add_recessions(ax1, start_date='1990-01-01')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='lower left', fontsize=9, **legend_style())
add_annotation_box(ax1, 'SLOOS leads loan growth 2-4 quarters.\nThe pipeline is the signal.')
brand_fig(fig, 'The Credit Pipeline', subtitle='SLOOS vs C&I Loan Growth', source='Federal Reserve via FRED', data_date=sloos_inv.index[-1])
plt.show()

In [ ]:
# Chart 6: Transmission Gap
ann_x, ann_y = 0.22, 0.12

real_rate = fetch_db_level('DFII10', start='2000-01-01')
hy = fetch_db_level('BAMLH0A0HYM2', start='2000-01-01')
hy_inv = hy * -100
start = max(real_rate.index[0], hy_inv.index[0]).strftime('%Y-%m-%d')
real_rate = real_rate[real_rate.index >= start]
hy_inv = hy_inv[hy_inv.index >= start]

fig, ax1 = new_fig()
ax2 = ax1.twinx()
c1, c2 = THEME['primary'], THEME['secondary']
ax1.plot(real_rate.index, real_rate, color=c1, linewidth=2.0, label=f'10Y Real Rate ({real_rate.iloc[-1]:.2f}%)')
ax2.plot(hy_inv.index, hy_inv, color=c2, linewidth=2.5, label=f'HY OAS Inverted ({hy_inv.iloc[-1]:.0f} bps inv.)')
style_dual_ax(ax1, ax2, c1, c2)
set_xlim_to_data(ax1, real_rate.index)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
add_last_value_label(ax1, real_rate, color=c1, fmt='{:.2f}%', side='left')
add_last_value_label(ax2, hy_inv, color=c2, fmt='{:.0f}', side='right')
add_recessions(ax1, start_date='2000-01-01')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper right', fontsize=9, **legend_style())
add_annotation_box(ax1, 'When these diverge, policy\nis not transmitting.')
brand_fig(fig, 'The Transmission Gap', subtitle='Real Rates vs HY Spreads (Inverted)', source='FRED, ICE BofA', data_date=real_rate.index[-1])
plt.show()

In [ ]:
# Chart 7: VIX vs VVIX
ann_x, ann_y = 0.50, 0.92

vix = fetch_db_level('VIXCLS', start='2007-01-01').rolling(21, min_periods=5).mean()
vvix = fetch_db_level('VXVCLS', start='2007-01-01').rolling(21, min_periods=5).mean()

fig, ax1 = new_fig()
ax2 = ax1.twinx()
c1, c2 = THEME['primary'], THEME['secondary']
ax1.plot(vix.index, vix, color=c1, linewidth=2.0, label=f'VIX 21d MA ({vix.iloc[-1]:.1f})')
ax2.plot(vvix.index, vvix, color=c2, linewidth=2.5, label=f'VVIX 21d MA ({vvix.iloc[-1]:.1f})')
ax1.axhline(25, color=COLORS['venus'], linewidth=1.0, linestyle=':', alpha=0.6)
style_dual_ax(ax1, ax2, c1, c2)
set_xlim_to_data(ax1, vix.index)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
add_last_value_label(ax1, vix, color=c1, fmt='{:.1f}', side='left')
add_last_value_label(ax2, vvix, color=c2, fmt='{:.1f}', side='right')
add_recessions(ax1, start_date='2007-01-01')
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper right', fontsize=9, **legend_style())
add_annotation_box(ax1, 'When VVIX rises faster than VIX,\ndealers are hedging tail risk.')
brand_fig(fig, 'VIX vs VVIX', subtitle='Tail Risk Detection', source='CBOE via FRED', data_date=vix.index[-1])
plt.show()

In [ ]:
# Chart 10: HY/IG Ratio
ann_x, ann_y = 0.82, 0.45

hy = fetch_db_level('BAMLH0A0HYM2', start='1997-01-01')
ig = fetch_db_level('BAMLC0A0CM', start='1997-01-01')
combined = pd.DataFrame({'hy': hy, 'ig': ig}).dropna()
ratio = (combined['hy'] / combined['ig']).rolling(21, min_periods=5).mean()

fig, ax = new_fig()
ax.plot(ratio.index, ratio, color=THEME['primary'], linewidth=2.0, label=f'HY/IG Ratio ({ratio.iloc[-1]:.1f}x)')
ax.axhline(ratio.median(), color=COLORS['fog'], linewidth=1.0, linestyle='--', alpha=0.7)
ax.axhline(5.0, color=COLORS['venus'], linewidth=1.5, linestyle='--', alpha=0.8)
ax.fill_between(ratio.index, 5.0, ratio, where=(ratio >= 5.0), color=COLORS['port'], alpha=0.15)
style_single_ax(ax, fmt='{:.1f}x')
set_xlim_to_data(ax, ratio.index)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
add_last_value_label(ax, ratio, color=THEME['primary'], fmt='{:.1f}x', side='right')
add_recessions(ax, start_date='1997-01-01')
ax.legend(loc='upper right', fontsize=9, **legend_style())
add_annotation_box(ax, 'Rising ratio = market differentiating.\nSpikes precede crises.')
brand_fig(fig, 'Credit Quality Differentiation', subtitle='HY/IG Ratio', source='ICE BofA via FRED', data_date=ratio.index[-1])
plt.show()

In [ ]:
# Chart 12: Stress Convergence
ann_x, ann_y = 0.50, 0.12

import numpy as np
def rolling_z(s, window=1260):
    m = s.rolling(window, min_periods=252).mean()
    sd = s.rolling(window, min_periods=252).std().replace(0, np.nan)
    return ((s - m) / sd).rolling(63, min_periods=10).mean()

baa_z = rolling_z(fetch_db_level('BAA10Y', start='1997-01-01') * 100)
nfci_z = rolling_z(fetch_db_level('NFCI', start='1997-01-01'))
vix_z = rolling_z(fetch_db_level('VIXCLS', start='1997-01-01'))
baa_z = baa_z[baa_z.index >= '2000-01-01']
nfci_z = nfci_z[nfci_z.index >= '2000-01-01']
vix_z = vix_z[vix_z.index >= '2000-01-01']

fig, ax = new_fig()
ax.plot(baa_z.index, baa_z, color=COLORS['ocean'], linewidth=2.0, label='Baa-10Y (z)', zorder=5)
ax.plot(nfci_z.index, nfci_z, color=COLORS['dusk'], linewidth=2.0, label='NFCI (z)', zorder=4)
ax.plot(vix_z.index, vix_z, color=COLORS['sky'], linewidth=1.5, label='VIX (z)', alpha=0.8, zorder=3)
ax.axhline(0, color=COLORS['fog'], linewidth=1.0, linestyle='--', alpha=0.7)
ax.axhline(1.5, color=COLORS['venus'], linewidth=1.0, linestyle=':', alpha=0.6)
style_single_ax(ax, fmt='{:.1f}')
set_xlim_to_data(ax, baa_z.index)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
add_recessions(ax, start_date='2000-01-01')
ax.legend(loc='upper left', fontsize=9, **legend_style())
add_last_value_label(ax, baa_z, color=COLORS['ocean'], fmt='{:.1f}', side='right')
add_annotation_box(ax, 'Convergence above zero = broad stress.\nDivergence = isolated.')
brand_fig(fig, 'Stress Convergence', subtitle='3 Public Signals Normalized', source='Chicago Fed, CBOE, Moody\'s via FRED', data_date=baa_z.index[-1])
plt.show()

## How to use

1. Change `ann_x` and `ann_y` in any cell above (0.0 = left/bottom, 1.0 = right/top)
2. Re-run that cell to see the result inline
3. Once happy, update the same values in `financial_edu_charts.py` and run `--all` to save PNGs